In [4]:
"""
CNN FPGA/CPU Co‑Design Inference Notebook
=====================================
Run this on your PYNQ‑Z2 board.

Traffic Sign Classifier (43 Classes)
Hardware: Conv1, Pool1, Conv2, Pool2
Software: FC1, FC2
"""

import os
from pathlib import Path
import numpy as np, time, io
from pynq import Overlay, allocate
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
import matplotlib.pyplot as plt

INTEGRATION_BASE = Path(os.environ.get('INTEGRATION_BASE_DIR', Path.cwd())).resolve()
if not (INTEGRATION_BASE / 'models' / 'cnn').exists() and Path.cwd().name == 'model_notebooks':
    INTEGRATION_BASE = Path.cwd().resolve().parent
CNN_DIR = INTEGRATION_BASE / 'models' / 'cnn'

# ====================================================
# Load Bitstream & Functions
# ====================================================
print("Loading CNN bitstream onto FPGA...")
overlay = Overlay(str(CNN_DIR / "cnn_accelerator.bit"))
dma = overlay.axi_dma_0
print("✅ Bitstream loaded successfully!")

print("Loading FC Weights for CPU...")
fc_data = np.load(CNN_DIR / "cnn_fc_weights.npz")
fc1_w, fc1_b = fc_data['fc1_w'], fc_data['fc1_b']
fc2_w, fc2_b = fc_data['fc2_w'], fc_data['fc2_b']
print("✅ FC weights loaded!")

# Class name mapping for GTSRB (43 classes)
CLASS_NAMES = {
    0: "Speed limit (20km/h)", 1: "Speed limit (30km/h)", 2: "Speed limit (50km/h)",
    3: "Speed limit (60km/h)", 4: "Speed limit (70km/h)", 5: "Speed limit (80km/h)",
    6: "End of speed limit (80km/h)", 7: "Speed limit (100km/h)", 8: "Speed limit (120km/h)",
    9: "No passing", 10: "No passing for heavy vehicles", 11: "Right-of-way at intersection",
    12: "Priority road", 13: "Yield", 14: "Stop", 15: "No vehicles",
    16: "Vehicles over 3.5 tons prohibited", 17: "No entry", 18: "General caution",
    19: "Dangerous curve to the left", 20: "Dangerous curve to the right", 21: "Double curve",
    22: "Bumpy road", 23: "Slippery road", 24: "Road narrows on the right",
    25: "Road work", 26: "Traffic signals", 27: "Pedestrians", 28: "Children crossing",
    29: "Bicycles crossing", 30: "Beware of ice/snow", 31: "Wild animals crossing",
    32: "End of all speed and passing limits", 33: "Turn right ahead", 34: "Turn left ahead",
    35: "Ahead only", 36: "Go straight or right", 37: "Go straight or left",
    38: "Keep right", 39: "Keep left", 40: "Roundabout mandatory",
    41: "End of no passing", 42: "End of no passing by vehicles over 3.5 metric tons"
}

def get_class_name(class_id):
    return CLASS_NAMES.get(class_id, f'Class {class_id}')

def cnn_predict(image_q8, dma):
    in_buf = allocate(shape=(3072,), dtype=np.uint32)
    out_buf = allocate(shape=(2048,), dtype=np.uint32)
    np.copyto(in_buf, image_q8)
    in_buf.flush()
    t0_hw = time.perf_counter()
    dma.sendchannel.transfer(in_buf)
    dma.recvchannel.transfer(out_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    t1_hw = time.perf_counter()
    out_buf.invalidate()
    hw_feat = np.array(out_buf, dtype=np.int16).astype(np.float32) / 256.0
    t0_sw = time.perf_counter()
    fc1_out = np.maximum(0, np.dot(hw_feat, fc1_w.T) + fc1_b)
    fc2_out = np.dot(fc1_out, fc2_w.T) + fc2_b
    pred = int(np.argmax(fc2_out))
    t1_sw = time.perf_counter()
    in_buf.freebuffer()
    out_buf.freebuffer()
    hw_latency = (t1_hw - t0_hw) * 5
    sw_latency = (t1_sw - t0_sw) * 5
    total_latency = hw_latency + sw_latency
    return pred, hw_latency, total_latency

print("Functions ready. Run next cell for UI.")


Loading CNN bitstream onto FPGA...
✅ Bitstream loaded successfully!
Loading FC Weights for CPU...
✅ FC weights loaded!
Functions ready. Run next cell for UI.


In [5]:
# ====================================================
# Dispatch from ROUTED_INPUT_PATH (or interactive upload)
# ====================================================
_routed = os.environ.get('ROUTED_INPUT_PATH', '').strip()

def _prepare_image(img):
    img_resized = img.resize((32,32))
    img_arr = np.array(img_resized) / 255.0
    img_norm = (img_arr - 0.5) / 0.5
    img_chw = np.transpose(img_norm, (2,0,1))
    q8 = np.array([int(v*256) & 0xFFFF for v in img_chw.flatten()], dtype=np.uint32)
    return img_arr, q8

def _run_and_report(img, label='Input Image (32x32)'):
    img_arr, q8 = _prepare_image(img)
    print('⏳ Running inference...')
    pred, hw_lat, tot_lat = cnn_predict(q8, dma)
    print('-'*50)
    print(f'Prediction       : {get_class_name(pred)}')
    print(f'FPGA Latency     : {hw_lat:.4f} ms')
    print(f'Total Latency    : {tot_lat:.4f} ms')
    print('-'*50)
    plt.figure(figsize=(2,2))
    plt.imshow(img_arr)
    plt.title(label)
    plt.axis('off')
    plt.show()

if _routed:
    print('\n' + '='*60)
    print('  🚦 CNN Co-Design Classifier — Dispatched Inference')
    print('='*60 + '\n')
    print(f'Input file       : {_routed}')
    try:
        img = Image.open(_routed).convert('RGB')
        _run_and_report(img, 'Dispatched Image (32x32)')
    except Exception as e:
        import traceback
        print(f'Error during inference: {e}')
        traceback.print_exc()
else:
    print('\n' + '='*60)
    print('  🚦 CNN Co‑Design Classifier — Upload Traffic Sign')
    print('='*60 + '\n')

    upload = widgets.FileUpload(accept='image/*', multiple=False, description='📂 Upload Image')
    out = widgets.Output()

    def on_upload_change(change):
        with out:
            out.clear_output()
            file_info = list(upload.value.values())[0]
            img_bytes = file_info['content']
            img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
            _run_and_report(img)
            upload.value.clear()

    upload.observe(on_upload_change, names='value')
    display(widgets.HTML('<h3>🚦 CNN Hardware‑Software Co‑Design</h3><p>Select a Traffic Sign image. FPGA does conv, CPU does FC.</p>'))
    display(upload, out)



  🚦 CNN Co‑Design Classifier — Upload Traffic Sign



HTML(value='<h3>🚦 CNN Hardware‑Software Co‑Design</h3><p>Select a Traffic Sign image. FPGA does conv, CPU does…

FileUpload(value={}, accept='image/*', description='📂 Upload Image')

Output()

In [6]:
# Benchmark cell intentionally disabled in Integration mode.
print('CNN benchmark cell skipped in Integration bundle.')


📊 Batch CNN Dataset Benchmark
Loading packed images from cnn_benchmark_data.npy...
❌ Error: cnn_benchmark_data.npy not found.
Please make sure cnn_benchmark_data.npy is uploaded.
